In [4]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/competitions/signate_mufgcup2024/notebooks

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/competitions/signate_mufgcup2024/notebooks


In [2]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from datetime import datetime
from sklearn.preprocessing import LabelEncoder

In [3]:
# configの読み込み
CONFIG_FILE = '../configs/config.yaml'
with open(CONFIG_FILE, encoding="utf-8") as file:
    yml = yaml.safe_load(file)

In [5]:
DIR_INPUT = yml["SETTING"]["DIR_INPUT"]
DIR_INTERIM = yml["SETTING"]["DIR_INTERIM"]
DIR_FEATURE = yml["SETTING"]["DIR_FEATURE"]
DIR_FIGURE = yml["SETTING"]["DIR_FIGURE"]
DIR_HOME = yml["SETTING"]["DIR_HOME"]
DIR_LOG = yml["SETTING"]["DIR_LOG"]
DIR_SUBMISSION = yml["SETTING"]["DIR_SUBMISSION"]
DIR_MODEL = yml["SETTING"]["DIR_MODEL"]
FILE_NAME_STATION = yml["SETTING"]["FILE_NAME_STATION"]
FILE_NAME_STATUS = yml["SETTING"]["FILE_NAME_STATUS"]
FILE_NAME_TRIP = yml["SETTING"]["FILE_NAME_TRIP"]
FILE_NAME_WEATHER = yml["SETTING"]["FILE_NAME_WEATHER"]

In [6]:
# 自作モジュールの読み込み
import sys
sys.path.append(DIR_HOME)
from src.util import Logger, Util

# データ読み込み

In [7]:
dict_dtype_station = {
    "station_id": "int64",
    "lat": "float64",
    "long": "float64",
    "dock_count": "int64",
    "city": "object",
    "installation_date": "object",
}
dict_dtype_status = {
    "id": "int64",
    "year": "int64",
    "month": "int64",
    "day": "int64",
    "hour": "int64",
    "station_id": "int64",
    "bikes_available": "float64",
    "predict": "int64",
}
dict_dtype_trip = {
    "trip_id": "int64",
    "duration": "int64",
    "start_date": "object",
    "start_station_id": "int64",
    "end_date": "object",
    "end_station_id": "int64",
    "bike_id": "int64",
    "subscription_type": "object",
}
dict_dtype_weather = {
    "date": "object",
    "max_temperature": "int64",
    "mean_temperature": "int64",
    "min_temperature": "int64",
    "max_dew_point": "int64",
    "mean_dew_point": "int64",
    "min_dew_point": "int64",
    "max_humidity": "int64",
    "mean_humidity": "int64",
    "min_humidity": "int64",
    "max_sea_level_pressure": "float64",
    "mean_sea_level_pressure": "float64",
    "min_sea_level_pressure": "float64",
    "max_visibility": "int64",
    "mean_visibility": "int64",
    "min_visibility": "int64",
    "max_wind_Speed": "int64",
    "mean_wind_speed": "int64",
    "precipitation": "float64",
    "cloud_cover": "int64",
    "events": "object",
    "wind_dir_degrees": "int64",
}

In [8]:
df_station = pd.read_csv(os.path.join(DIR_INPUT, FILE_NAME_STATION), dtype=dict_dtype_station)
df_status = pd.read_csv(os.path.join(DIR_INPUT, FILE_NAME_STATUS), dtype=dict_dtype_status)
df_trip = pd.read_csv(os.path.join(DIR_INPUT, FILE_NAME_TRIP), dtype=dict_dtype_trip)
df_weather = pd.read_csv(os.path.join(DIR_INPUT, FILE_NAME_WEATHER), dtype=dict_dtype_weather)

# 前処理

In [9]:
def convert_datetime(df_station, df_status, df_trip, df_weather):
    df_station["installation_date"] = pd.to_datetime(df_station["installation_date"], format='%m/%d/%Y').dt.date
    df_status["datetime"] = pd.to_datetime(df_status[["year", "month", "day", "hour"]])
    df_trip["start_date"] = pd.to_datetime(df_trip["start_date"], format='%m/%d/%Y %H:%M')
    df_trip["end_date"] = pd.to_datetime(df_trip["end_date"], format='%m/%d/%Y %H:%M')
    df_weather["date"] = pd.to_datetime(df_weather["date"], format='%Y-%m-%d').dt.date

    return df_station, df_status, df_trip, df_weather

In [10]:
df_station, df_status, df_trip, df_weather = convert_datetime(df_station, df_status, df_trip, df_weather)

In [11]:
# データ出力
df_station.to_pickle(os.path.join(DIR_INTERIM, "df_prep_station.pkl"))
df_status.to_pickle(os.path.join(DIR_INTERIM, "df_prep_status.pkl"))
df_trip.to_pickle(os.path.join(DIR_INTERIM, "df_prep_trip.pkl"))
df_weather.to_pickle(os.path.join(DIR_INTERIM, "df_prep_weather.pkl"))

# 特徴量作成

In [12]:
from pathlib import Path
from abc import ABCMeta, abstractmethod

In [13]:
from time import time

def decorate(s: str, decoration=None):
    if decoration is None:
        decoration = '★' * 20

    return ' '.join([decoration, str(s), decoration])

class Timer:
    def __init__(self, logger=None, format_str='{:.3f}[s]', prefix=None, suffix=None, sep=' ', verbose=0):

        if prefix: format_str = str(prefix) + sep + format_str
        if suffix: format_str = format_str + sep + str(suffix)
        self.format_str = format_str
        self.logger = logger
        self.start = None
        self.end = None
        self.verbose = verbose

    @property
    def duration(self):
        if self.end is None:
            return 0
        return self.end - self.start

    def __enter__(self):
        self.start = time()

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.end = time()
        if self.verbose is None:
            return
        out_str = self.format_str.format(self.duration)
        if self.logger:
            self.logger.info(out_str)
        else:
            print(out_str)

In [14]:
class AbstractBaseBlock(metaclass=ABCMeta):

    def __init__(self, use_cache=False, save_cache=False, logger=None):
        self.use_cache = use_cache
        self.name = self.__class__.__name__
        self.cache_dir = Path(DIR_FEATURE)
        self.logger = logger
        self.seve_cache = save_cache
        self.use_cols = None

    # 内部状態の更新
    def fit(self, df_input: pd.DataFrame, y=None):
        pass

    # 変換処理
    @abstractmethod
    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        raise NotImplementedError()

    # 内部状態の更新と変換処理をまとめて行う
    def fit_transform(self, X: pd.DataFrame, y=None):
        self.fit(X, y)
        return self.transform(X)

    # 特徴量生成処理
    def create_feature(self, X, y=None, test=False, limit_date="yyyy-mm-dd") -> pd.DataFrame:

        # クラス名.pkl
        file_name = os.path.join(self.cache_dir, limit_date, f"{self.name}.pkl")

        # キャッシュを使う & ファイルがあるなら読み出し
        if os.path.isfile(str(file_name)) and self.use_cache:
            return pd.read_pickle(file_name)

        # 変換処理を実行
        else:
            # trainの場合
            if not test:
                feature = self.fit_transform(X, y)
            # testの場合
            else:
                feature = self.transform(X)
            # 保存する場合
            if self.seve_cache:
                feature.to_pickle(file_name)

            return feature

In [15]:
# そのままの特徴量を返すBlock

In [16]:
# 静的特徴量
# 動的特徴量

In [17]:
class AsIsNumetricBlock(AbstractBaseBlock):

    def __init__(self, use_cache=False, save_cache=False, logger=None):
        super().__init__(use_cache, save_cache, logger)
        self.use_cols = [
            'CityTier',
            'DurationOfPitch',
            'NumberOfPersonVisiting',
            'NumberOfFollowups',
            'PreferredPropertyStar',
            'NumberOfTrips',
            'Passport',
            'PitchSatisfactionScore',
            'MonthlyIncome',
            'ProdTaken' # 目的関数
        ]
        self.key_col = ["id"]
        self.map_count = None

    def transform(self, df_input):
        df_out = pd.DataFrame()
        df_out = df_input[self.key_col + self.use_cols]
        return df_out

class AsIsCategoryBlock(AbstractBaseBlock):

    def __init__(self, use_cache=False, save_cache=False, logger=None):
        super().__init__(use_cache, save_cache, logger)
        self.use_cols = [
            'Age',
            'TypeofContact',
            'Occupation',
            'Gender',
            'ProductPitched',
            'Designation',
            'customer_info',
            'marry',
            'car',
            'child',
        ]
        self.key_col = ["id"]
        self.map_count = None

    def transform(self, df_input):
        df_out = pd.DataFrame()
        df_out = df_input[self.key_col + self.use_cols]
        return df_out

In [18]:
class CountEncodingBlock(AbstractBaseBlock):

    def __init__(self, use_cache=False, save_cache=False, logger=None):
        super().__init__(use_cache, save_cache, logger)
        self.use_cols = [
            "TypeofContact", "CityTier", "Occupation", "ProductPitched",
            "Designation", "marry", "car"
        ]
        self.key_col = ["id"]
        self.map_count = None

    def fit(self, df_input, y=None):
        # カラムごとにマッピング表を作成
        self.map_count = {}
        for col in self.use_cols:
            self.map_count[col] = df_input[col].fillna("NA").value_counts()

    def transform(self, df_input):
        df_out = pd.DataFrame()
        df_out[self.key_col] = df_input[self.key_col]
        for col in self.use_cols:
            df_out[col] = df_input[col].fillna("NA").map(self.map_count[col]).astype(int)
        # id以外のカラム名に接頭辞を付ける
        df_result = pd.concat([df_out[self.key_col], df_out.drop(columns=self.key_col).add_prefix('CE_')], axis=1)
        return df_result

class TargetEncodingBlock(AbstractBaseBlock):

    def __init__(self, use_cache=False, save_cache=False, logger=None):
        super().__init__(use_cache, save_cache, logger)
        self.use_cols = [
            "TypeofContact", "CityTier", "Occupation", "ProductPitched",
            "Designation", "marry", "car"
        ]
        self.key_col = ["id"]
        self.map_target = None

    def fit(self, df_input, y):
        # カラムごとにマッピング表を作成
        self.map_target = {}
        for col in self.use_cols:
            self.map_target[col] = df_input.groupby(col)[TARGET_COL].mean()

    def transform(self, df_input):
        df_out = pd.DataFrame()
        df_out[self.key_col] = df_input[self.key_col]
        for col in self.use_cols:
            df_out[col] = df_input[col].map(self.map_target[col]).astype(float)
        # id以外のカラム名に接頭辞を付ける
        df_result = pd.concat([df_out[self.key_col], df_out.drop(columns=self.key_col).add_prefix('TE_')], axis=1)
        return df_result

# 大分類を作成するクラス
class BigCategoryBlock(AbstractBaseBlock):

    def __init__(self, use_cache=False, save_cache=False, logger=None):
        super().__init__(use_cache, save_cache, logger)
        self.use_cols = []
        self.key_col = ["id"]
        self.map_target = None

    def fit(self, df_input, y=None):
        pass

    def transform(self, df_input):
        df_out = pd.DataFrame()
        df_out[self.key_col] = df_input[self.key_col]
        df_out["Age"] = df_input["Age"].apply(lambda x: self.replace_sai_to_dai(x) if not pd.isna(x) else x)
        # id以外のカラム名に接頭辞を付ける
        df_result = pd.concat([df_out[self.key_col], df_out.drop(columns=self.key_col).add_prefix('BC_')], axis=1)
        return df_result

    def replace_sai_to_dai(self, age):
        first_char = age[0]
        if first_char == "1":
            result = "10代"
        elif first_char == "2":
            result = "20代"
        elif first_char == "3":
            result = "30代"
        elif first_char == "4":
            result = "40代"
        elif first_char == "5":
            result = "50代"
        elif first_char == "6":
            result = "60代"
        return result

# 大分類を作成するクラス
class BigCategoryBlock(AbstractBaseBlock):

    def __init__(self, use_cache=False, save_cache=False, logger=None):
        super().__init__(use_cache, save_cache, logger)
        self.use_cols = []
        self.key_col = ["id"]
        self.map_target = None

    def fit(self, df_input, y=None):
        pass

    def transform(self, df_input):
        df_out = pd.DataFrame()
        df_out[self.key_col] = df_input[self.key_col]
        df_out["Age"] = df_input["Age"].apply(lambda x: self.replace_sai_to_dai(x) if not pd.isna(x) else x)
        # id以外のカラム名に接頭辞を付ける
        df_result = pd.concat([df_out[self.key_col], df_out.drop(columns=self.key_col).add_prefix('BC_')], axis=1)
        return df_result

    def replace_sai_to_dai(self, age):
        first_char = age[0]
        if first_char == "1":
            result = "10代"
        elif first_char == "2":
            result = "20代"
        elif first_char == "3":
            result = "30代"
        elif first_char == "4":
            result = "40代"
        elif first_char == "5":
            result = "50代"
        elif first_char == "6":
            result = "60代"
        return result

In [19]:
def run_blocks(df_input, feature_blocks, y=None, test=False):
    df_out = None

    print(decorate('start run blocks...'))

    with Timer(prefix='run test={}'.format(test)):
        for block in feature_blocks:
            with Timer(prefix='\t- {}'.format(str(block))):
                feature = block.create_feature(df_input, y=y, test=test)
            assert len(df_input) == len(feature), block
            if df_out is None:
                df_out = feature
            else:
                df_out = pd.merge(df_out, feature, on=["id"], how="left")

    return df_out

In [20]:
feature_blocks = [
    *[AsIsCategoryBlock(use_cache=False, save_cache=True, logger=None)],
    *[AsIsNumetricBlock(use_cache=True, save_cache=True, logger=None)],
    *[CountEncodingBlock(use_cache=True, save_cache=True, logger=None)],
    *[TargetEncodingBlock(use_cache=True, save_cache=True, logger=None)],
    *[BigCategoryBlock(use_cache=True, save_cache=True, logger=None)],
]

In [21]:
df_preprocessed = pd.read_pickle(os.path.join(DIR_INTERIM, "preprocessed.pkl"))

FileNotFoundError: [Errno 2] No such file or directory: '../data/interim/preprocessed.pkl'

In [ ]:
df_out = run_blocks(df_preprocessed, feature_blocks, y=None, test=False)

# 特徴量生成

In [ ]:
df_prep_status = pd.read_pickle(os.path.join(DIR_INTERIM, "df_prep_status.pkl"))
df_prep_trip = pd.read_pickle(os.path.join(DIR_INTERIM, "df_prep_trip.pkl"))
df_prep_station = pd.read_pickle(os.path.join(DIR_INTERIM, "df_prep_station.pkl"))
df_prep_weather = pd.read_pickle(os.path.join(DIR_INTERIM, "df_prep_weather.pkl"))

In [ ]:
def save_feature(df, file_name):
    file_name = file_name + ".pkl"
    df.to_pickle(os.path.join(DIR_FEATURE, file_name))

def load_feature(file_name):
    file_name = file_name + ".pkl"
    return pd.read_pickle(os.path.join(DIR_FEATURE, file_name))

def assert_data(df, key):
    # キーの重複を確認
    assert df[key].duplicated().sum() == 0, "key duplicate"
    # 欠損値の確認
    print("missing value: ")
    print(df.isnull().sum())

def assert_and_save_feature(df, key, file_name):
    assert_data(df, key)
    save_feature(df, file_name)

In [ ]:
# keyデータの作成
df_key = df_prep_status[["station_id", "datetime"]].copy()
assert_and_save_feature(df_key, ["station_id", "datetime"], "df_key")

In [ ]:
# 目的変数
df_target = df_prep_status[["station_id", "datetime", "bikes_available"]].copy()
assert_and_save_feature(df_target, ["station_id", "datetime"], "df_target")

In [ ]:
# predictデータの作成
df_predict = df_prep_status[["station_id", "datetime", "predict"]].copy()
assert_and_save_feature(df_predict, ["station_id", "datetime"], "df_predict")

In [ ]:
# 欠損値補完
def create_fillna_feature(df, target_col):
    df_ = pd.merge(df, load_feature("df_predict"), on=["station_id", "datetime"], how="left")
    df_fillna = pd.DataFrame()
    for station_id in df_["station_id"].unique().tolist():
        temp_df = df_[df_["station_id"]==station_id]
        ## 前の値で欠損値補完
        temp_df = temp_df.ffill()
        ## predict==1のデータをnullにする
        temp_df.loc[temp_df["predict"]==1, target_col] = np.nan
        df_fillna = pd.concat([df_fillna,temp_df])[["station_id", "datetime",target_col]]
    return df_fillna

In [ ]:
df_bikes_available_fillna = create_fillna_feature(df_target, "bikes_available")
assert_and_save_feature(df_bikes_available_fillna, ["station_id", "datetime"], "df_bikes_available_fillna")

In [ ]:
# ラグ特徴量の作成
def create_lag_feature(df, key_cols, target_cols, lag):
    df_lag = df[key_cols].copy()
    for target_col in target_cols:
        for i in range(0, lag+1):
            df_shift = df[key_cols + [target_col]].copy()
            df_shift["datetime"] = df_shift["datetime"] + pd.Timedelta(hours=i)
            df_shift.columns = key_cols + [f"{target_col}_lag{i}"]
            df_lag = pd.merge(df_lag, df_shift, on=key_cols, how="left")
    return df_lag



In [ ]:
lag = 48
df_bikes_available_lag = df_bikes_available_fillna.copy()
df_bikes_available_lag = create_lag_feature(df_bikes_available_lag, ["station_id", "datetime"], ["bikes_available"], lag)
assert_and_save_feature(df_bikes_available_lag, ["station_id", "datetime"], "df_bikes_available_lag")

In [ ]:
# diff・需要・供給・予測データの作成
trip_data = df_prep_trip.copy()
trip_data['start_hour'] = pd.to_datetime(trip_data['start_date']).dt.floor('h')
trip_data['end_hour'] = pd.to_datetime(trip_data['end_date']).dt.floor('h')

## 出発（需要）を集計
## 過去1時間の需要を集計
departures = trip_data.groupby(['start_station_id', 'start_hour']).size().reset_index(name='demand')
departures["datetime"] = departures["start_hour"] + pd.Timedelta(hours=1)
departures.rename(columns={"start_station_id": "station_id"}, inplace=True)
## 到着（供給）を集計
## 過去1時間の供給を集計
arrivals = trip_data.groupby(['end_station_id', 'end_hour']).size().reset_index(name='supply')
arrivals["datetime"] = arrivals["end_hour"] + pd.Timedelta(hours=1)
arrivals.rename(columns={"end_station_id": "station_id"}, inplace=True)

## bikes_availableを需要・供給・介入に分解
df_status_detail = df_prep_status[["station_id", "datetime", "bikes_available"]].copy()
df_status_detail['datetime'] = pd.to_datetime(df_status_detail['datetime']).dt.floor('h')

## 前の時間との利用可能数の差分を計算
df_status_detail = df_status_detail.sort_values(['station_id', 'datetime'])
df_status_detail['bikes_available_diff'] = df_status_detail.groupby('station_id')['bikes_available'].diff()

## 需要・供給を結合
df_status_detail = df_status_detail.merge(departures, on=['station_id', 'datetime'], how='left')
df_status_detail = df_status_detail.merge(arrivals, on=['station_id', 'datetime'], how='left')
df_status_detail["demand"] = -df_status_detail["demand"]
df_status_detail["demand"] = df_status_detail["demand"].fillna(0)
df_status_detail["supply"] = df_status_detail["supply"].fillna(0)

## 介入を計算
## 過去１時間の介入を計算
df_status_detail['intervention'] = df_status_detail['bikes_available_diff'] - (df_status_detail['demand'] + df_status_detail['supply'])

df_demand = df_status_detail[['station_id', 'datetime', 'demand']].copy()
df_supply = df_status_detail[['station_id', 'datetime', 'supply']].copy()
df_intervention = df_status_detail[['station_id', 'datetime', 'intervention']].copy()
assert_and_save_feature(df_demand, ["station_id", "datetime"], "df_demand")
assert_and_save_feature(df_supply, ["station_id", "datetime"], "df_supply")
assert_and_save_feature(df_intervention, ["station_id", "datetime"], "df_intervention")

In [ ]:
# 欠損値補完
df_intervention_fillna = create_fillna_feature(df_intervention, "intervention")
assert_and_save_feature(df_intervention_fillna, ["station_id", "datetime"], "df_intervention_fillna")

In [ ]:
# 需要・供給・介入のラグ特徴量を作成
df_demand_lag = create_lag_feature(df_demand, ["station_id", "datetime"], ["demand"], 24)
df_supply_lag = create_lag_feature(df_supply, ["station_id", "datetime"], ["supply"], 24)
df_intervention_lag = create_lag_feature(df_intervention_fillna, ["station_id", "datetime"], ["intervention"], 24)
assert_and_save_feature(df_demand_lag, ["station_id", "datetime"], "df_demand_lag")
assert_and_save_feature(df_supply_lag, ["station_id", "datetime"], "df_supply_lag")
assert_and_save_feature(df_intervention_lag, ["station_id", "datetime"], "df_intervention_lag")

In [ ]:
# 各日の00:00からのcumsumに変換
def calc_cumsum_from_00(df_, col):
    df_["date"] = df_["datetime"].dt.date
    df_[f"{col}_cumsum"] = df_.groupby(["station_id", "date"])[col].cumsum()
    df_ = df_.drop(columns=[col, "date"])
    return df_

df_demand_cumsum = calc_cumsum_from_00(df_demand, "demand")
df_supply_cumsum = calc_cumsum_from_00(df_supply, "supply")
df_intervention_cumsum = calc_cumsum_from_00(df_intervention_fillna, "intervention")
assert_and_save_feature(df_demand_cumsum, ["station_id", "datetime"], "df_demand_cumsum")
assert_and_save_feature(df_supply_cumsum, ["station_id", "datetime"], "df_supply_cumsum")
assert_and_save_feature(df_intervention_cumsum, ["station_id", "datetime"], "df_intervention_cumsum")

In [ ]:
# cumsumのラグ特徴量を作成
df_demand_cumsum_lag = create_lag_feature(df_demand_cumsum, ["station_id", "datetime"], ["demand_cumsum"], 24)
df_supply_cumsum_lag = create_lag_feature(df_supply_cumsum, ["station_id", "datetime"], ["supply_cumsum"], 24)
df_intervention_cumsum_lag = create_lag_feature(df_intervention_cumsum, ["station_id", "datetime"], ["intervention_cumsum"], 24)
assert_and_save_feature(df_demand_cumsum_lag, ["station_id", "datetime"], "df_demand_cumsum_lag")
assert_and_save_feature(df_supply_cumsum_lag, ["station_id", "datetime"], "df_supply_cumsum_lag")
assert_and_save_feature(df_intervention_cumsum_lag, ["station_id", "datetime"], "df_intervention_cumsum_lag")

In [ ]:
# 全ての目的変数を作成
df_target_all = df_target.copy()
df_target_all["demand"] = df_status_detail["demand"]
df_target_all["supply"] = df_status_detail["supply"]
df_target_all["intervention"] = df_status_detail["intervention"]
df_target_all["demand_cumsum"] = df_demand_cumsum["demand_cumsum"]
df_target_all["supply_cumsum"] = df_supply_cumsum["supply_cumsum"]
df_target_all["intervention_cumsum"] = df_intervention_cumsum["intervention_cumsum"]
df_target_all["demand_suplly_cumsum"] = df_target_all["demand_cumsum"] + df_target_all["supply_cumsum"]
assert_and_save_feature(df_target_all, ["station_id", "datetime"], "df_target_all")

In [ ]:
# predictの検証データ用のフラグを立てる
def create_valid_flag(df_result):
    # テストデータと2日以上離れているデータを検証データとする
    df_ = df_result.copy()
    df_ = df_[df_["predict"] != 0]
    df_["date"] = df_["datetime"].dt.date
    df_ = df_[["date","predict"]].drop_duplicates()
    df_["diff"] = df_["date"].diff().dt.days
    df_["yesterday"] = df_["date"].apply(lambda x: x - pd.Timedelta(days=1))
    list_valid_date = df_[df_["diff"] >= 4.0]["yesterday"].tolist()
    # dateがlist_valid_dateに含まれている場合はpredictを2にする
    df_result_ = df_result.copy()
    df_result_["predict"] = df_result_.apply(lambda x: 2 if (x["datetime"].date() in list_valid_date) & (x["datetime"].hour!=0) else x["predict"], axis=1)
    return df_result_

def create_add_valid_dataset(df_main):
    """検証データのフラグを立てる
    """
    df_main_ = df_main.copy()
    df_main_ = df_main_.sort_values(["datetime","station_id"],ascending=True).reset_index(drop=True)
    past_2_len = 0
    while True:
        df_main_ = create_valid_flag(df_main_)
        current_2_len = len(df_main_[df_main_["predict"] == 2])
        if current_2_len == past_2_len:
            break
        past_2_len = current_2_len

    return df_main_

df_predict_add_valid_flag = create_add_valid_dataset(df_predict)
assert_and_save_feature(df_predict_add_valid_flag, ["station_id", "datetime"], "df_predict_add_valid_flag")
print(df_predict_add_valid_flag.groupby("predict").size())

In [ ]:
# stationに関する特徴量を作成
# keyはstation_id
df_station_atr = df_prep_station.copy()
# staion_idの特徴量を追加
df_station_atr["station_id_feat"] = df_station_atr["station_id"]
# installation_dateに関する特徴量を追加
df_station_atr["installation_year"] = df_station_atr["installation_date"].apply(lambda x: x.year)
df_station_atr["installation_month"] = df_station_atr["installation_date"].apply(lambda x: x.month)
df_station_atr["installation_day"] = df_station_atr["installation_date"].apply(lambda x: x.day)
df_station_atr["installation_date"] = df_station_atr["installation_date"].apply(lambda x: int(x.strftime('%Y%m%d')))
# ダミー変数化
df_station_atr = pd.get_dummies(df_station_atr, columns=['city'], dtype="int64")

assert_and_save_feature(df_station_atr, ["station_id"], "df_station_atr")

In [ ]:
# datetimeに関する特徴量を作成
df_datetime_atr = df_prep_status[["datetime"]].drop_duplicates().copy()
# datetime
df_datetime_atr["datetime_feat"] = df_datetime_atr["datetime"].apply(lambda x: int(x.strftime('%Y%m%d')))
# 年
df_datetime_atr["year"] = df_datetime_atr["datetime"].dt.year
# 月
df_datetime_atr["month"] = df_datetime_atr["datetime"].dt.month
# 日
df_datetime_atr["day"] = df_datetime_atr["datetime"].dt.day
# 時間
df_datetime_atr["hour"] = df_datetime_atr["datetime"].dt.hour
# 曜日
df_datetime_atr["weekday"] = df_datetime_atr["datetime"].dt.weekday
# 休日か否かのフラグ
df_datetime_atr["holiday"] = df_datetime_atr["weekday"].apply(lambda x: 1 if x in [5, 6] else 0)

assert_and_save_feature(df_datetime_atr, ["datetime"], "df_datetime_atr")

In [ ]:
# weatherに関する特徴量を作成
df_weather_feat = df_key[["datetime"]].drop_duplicates().copy()
# datetimeに拡張
df_weather_feat["date"] = df_weather_feat["datetime"].dt.date
df_weather_feat = pd.merge(df_weather_feat, df_prep_weather, on=["date"], how="left")
df_weather_feat = df_weather_feat.drop(columns=["date"])
# ダミー変数化
df_weather_feat = pd.get_dummies(df_weather_feat, columns=['events'], drop_first=True, dtype="int64")

assert_and_save_feature(df_weather_feat, ["datetime"], "df_weather_feat")

In [ ]:
# 遅延特徴量

In [ ]:
# 移動の作成
def create_rolling_feature(df, key_cols, target_col, window_sizes, group_col="station_id"):
    df_rolling = df[key_cols+[target_col]].copy()
    for window in window_sizes:
        # 移動平均
        av_col_name = f"{target_col}_av{window}"
        df_rolling[av_col_name] = df_rolling.groupby(group_col)[target_col].transform(lambda x: x.rolling(window=window, min_periods=1).mean())
        # 移動標準偏差
        std_col_name = f"{target_col}_std{window}"
        df_rolling[std_col_name] = df_rolling.groupby(group_col)[target_col].transform(lambda x: x.rolling(window=window, min_periods=1).std())
        # 移動最大値
        max_col_name = f"{target_col}_max{window}"
        df_rolling[max_col_name] = df_rolling.groupby(group_col)[target_col].transform(lambda x: x.rolling(window=window, min_periods=1).max())
        # 移動最小値
        min_col_name = f"{target_col}_min{window}"
        df_rolling[min_col_name] = df_rolling.groupby(group_col)[target_col].transform(lambda x: x.rolling(window=window, min_periods=1).min())
        # 移動変動係数
        cv_col_name = f"{target_col}_cv{window}"
        df_rolling[cv_col_name] = df_rolling[std_col_name] / df_rolling[av_col_name]
    df_rolling = df_rolling.drop(columns=[target_col])
    return df_rolling

In [ ]:
window_sizes = [3, 7, 24, 48]
df_bikes_available_rolling = create_rolling_feature(df_bikes_available_fillna, ["station_id", "datetime"], "bikes_available", window_sizes)
assert_and_save_feature(df_bikes_available_rolling, ["station_id", "datetime"], "df_bikes_available_rolling")
df_demand_rolling = create_rolling_feature(df_demand, ["station_id", "datetime"], "demand", window_sizes)
assert_and_save_feature(df_demand_rolling, ["station_id", "datetime"], "df_demand_rolling")
df_supply_rolling = create_rolling_feature(df_supply, ["station_id", "datetime"], "supply", window_sizes)
assert_and_save_feature(df_supply_rolling, ["station_id", "datetime"], "df_supply_rolling")
df_intervention_rolling = create_rolling_feature(df_intervention_fillna, ["station_id", "datetime"], "intervention", window_sizes)
assert_and_save_feature(df_intervention_rolling, ["station_id", "datetime"], "df_intervention_rolling")

In [ ]:
# 欠損値補完
df_bikes_available_rolling_fillna = load_feature("df_bikes_available_rolling")
df_bikes_available_rolling_fillna = df_bikes_available_rolling_fillna.ffill()
assert_and_save_feature(df_bikes_available_rolling_fillna, ["station_id", "datetime"], "df_bikes_available_rolling_fillna")

In [ ]:
# 増減率の作成
def create_increase_rate_feature(df, key_cols, target_col, window_sizes, group_col="station_id"):
    df_rate = df[key_cols+[target_col]].copy()
    for window in window_sizes:
        # 増減率
        rate_col_name = f"{target_col}_ir{window}"
        df_rate[rate_col_name] = df_rate.groupby(group_col)[target_col].pct_change(window, fill_method=None)
    df_rate = df_rate.drop(columns=[target_col])
    return df_rate

In [ ]:
window_sizes = [3, 7, 24, 48]
df_bikes_available_lr = create_increase_rate_feature(df_bikes_available_fillna, ["station_id", "datetime"], "bikes_available", window_sizes)
assert_and_save_feature(df_bikes_available_lr, ["station_id", "datetime"], "df_bikes_available_lr")
df_demand_lr = create_increase_rate_feature(df_demand, ["station_id", "datetime"], "demand", window_sizes)
assert_and_save_feature(df_demand_lr, ["station_id", "datetime"], "df_demand_lr")
df_supply_lr = create_increase_rate_feature(df_supply, ["station_id", "datetime"], "supply", window_sizes)
assert_and_save_feature(df_supply_lr, ["station_id", "datetime"], "df_supply_lr")
df_intervention_lr = create_increase_rate_feature(df_intervention_fillna, ["station_id", "datetime"], "intervention", window_sizes)
assert_and_save_feature(df_intervention_lr, ["station_id", "datetime"], "df_intervention_lr")

# 欠損値補完
df_bikes_available_lr_fillna = load_feature("df_bikes_available_lr")
df_bikes_available_lr_fillna = df_bikes_available_lr_fillna.ffill()
assert_and_save_feature(df_bikes_available_lr_fillna, ["station_id", "datetime"], "df_bikes_available_lr_fillna")

In [ ]:
# 欠損値補完
df_bikes_available_rolling_fillna = load_feature("df_bikes_available_rolling")
for col in df_bikes_available_rolling_fillna.columns:
    df_bikes_available_rolling_fillna = create_fillna_feature(df_bikes_available_rolling_fillna, "df_bikes_available_rolling")
assert_and_save_feature(df_bikes_available_fillna, ["station_id", "datetime"], "df_bikes_available_rolling_fillna")

In [ ]:
# dock_count_rateの作成
df_bikes_available_rate = load_feature("df_bikes_available_fillna").copy()
df_bikes_available_rate = pd.merge(df_bikes_available_rate, df_station_atr[["station_id", "dock_count"]], on="station_id", how="left")
df_bikes_available_rate["dock_count_rate"] = df_bikes_available_rate["bikes_available"] / df_bikes_available_rate["dock_count"]
df_bikes_available_rate = df_bikes_available_rate.drop(columns=["bikes_available", "dock_count"])
assert_and_save_feature(df_bikes_available_rate, ["station_id", "datetime"], "df_bikes_available_rate")

In [ ]:
# dock_count_rateのラグ特徴量を作成
df_bikes_available_rate_lag = create_lag_feature(df_bikes_available_rate, ["station_id", "datetime"], ["dock_count_rate"], 24)
assert_and_save_feature(df_bikes_available_rate_lag, ["station_id", "datetime"], "df_bikes_available_rate_lag")

In [ ]:
# demand_cumsumをプロット
# 期間を絞る
df_ = df_target_all.copy()
col = "demand_suplly_cumsum"
station_id = 0
start_date = "2014-09-16"
end_date = "2014-10-18"
df_ = df_[df_["station_id"]==station_id]
df_ = df_[(df_["datetime"] >= start_date) & (df_["datetime"] <= end_date)]
plt.figure(figsize=(20, 10))
plt.plot(df_["datetime"], df_[col])
plt.show()

In [ ]:
# tripデータでbikes_availableを補完する
df_key = load_feature("df_key")
df_bikes_available_fillna = load_feature("df_bikes_available_fillna")
df_demand_cumsum = load_feature("df_demand_cumsum")
df_supply_cumsum = load_feature("df_supply_cumsum")

df_bikes_trip = df_key.copy()
df_bikes_trip = pd.merge(df_bikes_trip, df_demand_cumsum, on=["station_id", "datetime"], how="left")
df_bikes_trip = pd.merge(df_bikes_trip, df_supply_cumsum, on=["station_id", "datetime"], how="left")

df_bikes_00 = df_bikes_available_fillna[df_bikes_available_fillna["datetime"].dt.hour==0][["station_id", "datetime", "bikes_available"]].copy()
df_bikes_00["date"] = df_bikes_00["datetime"].dt.date
df_bikes_00 = df_bikes_00.rename(columns={"bikes_available": "bikes_available_00"})
df_bikes_00 = df_bikes_00[["station_id", "date", "bikes_available_00"]]

df_bikes_trip["date"] = df_bikes_trip["datetime"].dt.date
df_bikes_trip = pd.merge(df_bikes_trip, df_bikes_00, on=["station_id", "date"], how="left")
df_bikes_trip = df_bikes_trip.drop(columns=["date"])

# 補完したデータにはフラグを立てる
df_bikes_trip["bikes_available"] = df_bikes_trip["bikes_available_00"] + df_bikes_trip["demand_cumsum"] + df_bikes_trip["supply_cumsum"]
df_bikes_trip = df_bikes_trip[["station_id", "datetime", "bikes_available"]]
df_bikes_available_trip_fillna = pd.merge(df_bikes_available_fillna, df_bikes_trip, on=["station_id", "datetime"], how="left", suffixes=("", "_pred"))
df_bikes_available_trip_fillna_00 = df_bikes_available_trip_fillna.copy()
df_bikes_available_trip_fillna_00["is_trip_fillna"] = df_bikes_available_trip_fillna_00.apply(lambda x: 1 if pd.isna(x["bikes_available"]) else 0, axis=1).shift(-1)
df_bikes_available_trip_fillna_00 = df_bikes_available_trip_fillna_00[df_bikes_available_trip_fillna_00["datetime"].dt.hour==0]
df_bikes_available_trip_fillna = pd.merge(df_bikes_available_trip_fillna, df_bikes_available_trip_fillna_00[["station_id", "datetime", "is_trip_fillna"]], on=["station_id", "datetime"], how="left")
df_bikes_available_trip_fillna["is_trip_fillna"] = df_bikes_available_trip_fillna["is_trip_fillna"].fillna(0)
df_bikes_available_trip_fillna["bikes_available"] = df_bikes_available_trip_fillna.apply(lambda x: x["bikes_available_pred"] if pd.isna(x["bikes_available"]) else x["bikes_available"], axis=1)
df_bikes_available_trip_fillna = df_bikes_available_trip_fillna.drop(columns=["bikes_available_pred"])

assert_and_save_feature(df_bikes_available_trip_fillna[["station_id","datetime","bikes_available"]], ["station_id", "datetime"], "df_bikes_available_trip_fillna")
assert_and_save_feature(df_bikes_available_trip_fillna[["station_id","datetime","is_trip_fillna"]], ["station_id", "datetime"], "df_is_trip_fillna")

In [ ]:
# ラグ特徴量の作成
lag = 48
df_bikes_available_lag = load_feature("df_bikes_available_trip_fillna")
df_bikes_available_lag = create_lag_feature(df_bikes_available_lag, ["station_id", "datetime"], ["bikes_available"], lag)
assert_and_save_feature(df_bikes_available_lag, ["station_id", "datetime"], "df_bikes_available_trip_fillna_lag")

In [ ]:
#

In [ ]:
def get_cv_folds(freq='MS'):
    """CVのfoldを返却
    """
    return pd.date_range("2014-09-01", periods=12, freq=freq)

In [ ]:
get_cv_folds("3MS")

In [ ]:
df_predict_add_valid_flag = load_feature("df_predict_add_valid_flag")
for date in pd.date_range(start="2014-09-01", end="2015-08-01", freq="3MS"):
    print(date)
    print(date + pd.DateOffset(months=3))

In [ ]:
from sklearn.metrics import mean_absolute_error
def metric(va_true, va_pred):
    """評価指標の計算
    """
    score = mean_absolute_error(va_true, va_pred)
    # score = math.sqrt(mean_squared_error(va_true, va_pred))

    return score

In [ ]:
run_name = "lgbm_multimodel_bikes_202410201528"
df_va_pred = pd.read_pickle(os.path.join(DIR_MODEL, f"{run_name}/va_pred.pkl"))
df_true = load_feature("df_target")
df_true_pred = pd.merge(df_true, df_va_pred, on=["station_id", "datetime"], how="inner", suffixes=("", "_pred"))
df_true_pred["weekdays"] = df_true_pred["datetime"].dt.weekday
df_true_pred_holiday = df_true_pred[df_true_pred["weekdays"].isin([5, 6])]
df_true_pred_weekday = df_true_pred[~df_true_pred["weekdays"].isin([5, 6])]

print("holiday")
print(metric(df_true_pred_holiday["bikes_available"], df_true_pred_holiday["bikes_available_pred"]))
print("weekday")
print(metric(df_true_pred_weekday["bikes_available"], df_true_pred_weekday["bikes_available_pred"]))

In [ ]:
df_true_pred_staion = pd.merge(df_true_pred, load_feature("df_station_atr"), on="station_id", how="left")
df_true_pred_city1 = df_true_pred_staion[df_true_pred_staion["city_city1"]==1]
df_true_pred_city2 = df_true_pred_staion[df_true_pred_staion["city_city2"]==1]
df_true_pred_city3 = df_true_pred_staion[df_true_pred_staion["city_city3"]==1]
df_true_pred_city4 = df_true_pred_staion[df_true_pred_staion["city_city4"]==1]
df_true_pred_city5 = df_true_pred_staion[df_true_pred_staion["city_city5"]==1]

print("city1", df_true_pred_city1.shape)
print(metric(df_true_pred_city1["bikes_available"], df_true_pred_city1["bikes_available_pred"]))
print("city2", df_true_pred_city2.shape)
print(metric(df_true_pred_city2["bikes_available"], df_true_pred_city2["bikes_available_pred"]))
print("city3", df_true_pred_city3.shape)
print(metric(df_true_pred_city3["bikes_available"], df_true_pred_city3["bikes_available_pred"]))
print("city4", df_true_pred_city4.shape)
print(metric(df_true_pred_city4["bikes_available"], df_true_pred_city4["bikes_available_pred"]))
print("city5", df_true_pred_city5.shape)
print(metric(df_true_pred_city5["bikes_available"], df_true_pred_city5["bikes_available_pred"]))

In [ ]:
df_true_pred_city2